# Inspect one model

**Question: which checkpoint was evaluated, on which data, and where does it fail?**

1. Export a report with `python3 scripts/evaluate.py MODEL DATASET --output results/evaluation/NAME`.
2. Set `RUN`, `MODEL_LABEL` and `DATASET_ROLE` below, then run all cells.
3. Check the recorded model/data hashes before interpreting scores or exporting figures.

This notebook reads saved results; it does not train or run predictions.
Use validation for tuning. A test score belongs only to its evaluated checkpoint;
the project's test split has already been consulted for historical models.
Compare multiple validation runs in [comparison.ipynb](comparison.ipynb).


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN = None  # Set a result path, e.g. ROOT / "results/evaluation/NAME".
MODEL_LABEL = "Selected checkpoint"  # Human-readable name for figure titles.
DATASET_ROLE = "unspecified"  # Set to "train", "validation" or "test" from the run record.
PHASE = "validation"  # Nested training/experiment metrics: "before" or "validation".

if RUN is None:
    raise ValueError("Set RUN to a saved evaluation or training directory before running this notebook.")
RUN = Path(RUN)
if not RUN.is_absolute():
    RUN = ROOT / RUN
if not RUN.is_dir():
    raise FileNotFoundError(f"No result directory at {RUN}")
if DATASET_ROLE not in ("train", "validation", "test", "unspecified"):
    raise ValueError("DATASET_ROLE must be train, validation, test or unspecified.")

path = RUN / "metrics.json"
metrics = json.loads(path.read_text()) if path.is_file() else None
metrics_phase = "standalone" if metrics is not None else "history only"
if metrics is not None and "confusion_matrix" not in metrics:
    metrics_phase = PHASE
    if PHASE not in metrics:
        raise ValueError(f"No phase {PHASE!r} in {path}; select an available metrics phase.")
    metrics = metrics[PHASE]

TITLE = f"{MODEL_LABEL} | {DATASET_ROLE} | {metrics_phase}"
print(TITLE)
print(f"Run: {RUN}")
print(f"Recorded metrics: {path}")
if DATASET_ROLE == "unspecified":
    print("Dataset role is unspecified: confirm the run record before interpreting or sharing figures.")
if metrics is None:
    print("Training only: inspect curves below; export an evaluation for classification scores.")
else:
    labels = metrics["confusion_matrix"]["labels"]
    print(f"Positions: {metrics['samples']:,}")
    print(f"Accuracy: {metrics['accuracy']:.2%} | Macro F1: {metrics['macro_f1']:.2%}")
    print(f"Analyzer elapsed time: {metrics['prediction_seconds']:.2f} s (includes loading)")
    for name in ("model", "dataset", "analyzer"):
        artifact = metrics["artifacts"].get(name, {})
        print(f"{name.capitalize()}: {artifact.get('path', 'Path not retained in these metrics')}")
        print(f"  Recorded SHA-256: {artifact.get('sha256', 'Not recorded')}")
    print("Hashes identify the evaluated bytes; this notebook does not reverify current files.")


## Scores by class

Precision: how often predictions of a class are correct. Recall: how many actual
positions of that class were found. F1 combines both. Undefined scores are zero;
macro F1 includes all three classes, even those with no examples.


In [ ]:
if metrics is not None:
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    width = 0.24
    for offset, key in enumerate(("precision", "recall", "f1")):
        positions = [i + (offset - 1) * width for i in range(len(labels))]
        bars = ax.bar(positions, [metrics["per_class"][label][key] for label in labels],
                      width, label=key.capitalize())
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.set_xticks(range(len(labels)),
                  [f"{label}\nn={metrics['per_class'][label]['support']:,}" for label in labels])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title(f"Per-class performance — {TITLE}")
    ax.legend(loc="upper right", ncols=3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


## Confusion matrix

Rows are actual classes; columns are predictions. The diagonal shows correct
predictions. Row percentages make errors comparable across class sizes.
A row with no examples is shown as zeros.


In [ ]:
if metrics is not None:
    counts = metrics["confusion_matrix"]["counts"]
    percentages = [[100 * value / sum(row) if sum(row) else 0 for value in row]
                   for row in counts]
    fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
    heatmap = ax.imshow(percentages, cmap="Blues", vmin=0, vmax=100)
    for row in range(len(labels)):
        for column in range(len(labels)):
            percent = percentages[row][column]
            ax.text(column, row, f"{counts[row][column]:,}\n{percent:.1f}%",
                    ha="center", va="center", color="white" if percent > 50 else "black")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_title(f"Confusion matrix — {TITLE}")
    fig.colorbar(heatmap, ax=ax, label="% of actual class")
    plt.show()


## Training curves
Available when RUN points to a training directory or an experiment run.

In [ ]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.comparison import load_run_history

history = load_run_history(RUN)
if history:
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    for key, label in (("train_loss", "Train"), ("validation_loss", "Validation")):
        ax.plot([row["epoch"] for row in history], [row[key] for row in history], label=label)
    ax.set(xlabel="Epoch (including previous training when recorded)", ylabel="Loss", title=TITLE)
    ax.legend()
    plt.show()
else:
    print("Standalone evaluation: no training history in this folder.")


## Interpretation

- Confirm the checkpoint, recorded hashes, dataset role and metrics phase.
- Report precision/recall/F1 and the main confusion, alongside accuracy.
- Compare against always-Nothing on the same data; it misses every Check and Checkmate.
- For train/validation curves, distinguish the selected checkpoint from the final epoch.
- Record the observed weakness, a possible explanation and what would test it.

An untrained model only checks the pipeline. A reused validation score does not
establish performance on a new source, and labels in titles do not certify provenance.
